# Week 3 Assingnment - Customer Sales Insight using SQL
**Objective** Analyze sales data using SQL by applying Subqueries, CTEs, and Window Functions to solve business queries.


# Step 1: Setting the SQL environment


In [202]:
import pandas as pd
import sqlite3

df = pd.read_csv('Dataset/Sample - Superstore.csv',encoding = 'latin1')
con = sqlite3.connect(':memory:')
df.to_sql('superstore_raw',con,if_exists='replace',index=False)

def run_query(query):
    return pd.read_sql_query(query,con).head(10)

df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [203]:
print("Rows:",df.shape[0])
print("Columns:",df.shape[1])

Rows: 9994
Columns: 21


In [204]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [205]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

# Step-2 Creating 3 Tables from Superstore.csv 
We used 'SELECT DISTINCT' to create 3 tables from 'Superstore.csv'.
- **customers** 
- **products** 
- **orders** 

In [206]:
con.executescript('''
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;

CREATE TABLE customers AS
SELECT DISTINCT "Customer ID" AS customer_id, "Customer Name" AS customer_name,
  Segment AS segment, Country AS country, City AS city,
  State AS state, Region AS region
FROM superstore_raw;

CREATE TABLE products AS
SELECT DISTINCT "Product ID" AS product_id, Category AS category,
  "Sub-Category" AS sub_category, "Product Name" AS product_name
FROM superstore_raw;

CREATE TABLE orders AS
SELECT "Row ID" AS row_id, "Order ID" AS order_id, "Order Date" AS order_date,
  "Ship Date" AS ship_date, "Ship Mode" AS ship_mode,
  "Customer ID" AS customer_id, "Product ID" AS product_id,
  Sales AS sales, Quantity AS quantity, Discount AS discount, Profit AS profit
FROM superstore_raw;
''')
con.commit()

In [207]:
run_query("""SELECT * FROM customers LIMIT 5;""")


,customer_id,customer_name,segment,country,city,state,region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South


In [208]:
run_query("""SELECT * FROM products LIMIT 5;""")

,product_id,category,sub_category,product_name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


In [209]:
run_query("""SELECT * FROM orders LIMIT 5;""")

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


# Step 3: Perform Required Queries
Write and execute SQL queries for each of the following:
1. Find all orders where sales are greater than the average sales. (Subquery)  
2. Find the highest sales order for each customer. (Subquery)  
3. Calculate total sales for each customer. (CTE)  
4. Find customers whose total sales are above average. (CTE + Subquery)  
5. Rank all customers based on total sales. (Window Function)  
6. Assign row numbers to each order within a customer. (Window Function + PARTITION BY)  
7. Display top 3 customers based on total sales. (Window Function)  

In [210]:
# Ques1:
run_query("""SELECT order_id, customer_id, sales FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders) ORDER BY sales DESC;""")

,order_id,customer_id,sales
0,CA-2014-145317,SM-20320,22638.480
1,CA-2016-118689,TC-20980,17499.950
2,CA-2017-140151,RB-19360,13999.960
3,CA-2017-127180,TA-21385,11199.968
4,CA-2017-166709,HL-15040,10499.970
5,CA-2016-117121,AB-10105,9892.740
6,CA-2014-116904,SC-20095,9449.950
7,US-2016-107440,BS-11365,9099.930
8,CA-2016-158841,SE-20110,8749.950
9,CA-2016-143714,CC-12370,8399.976


In [211]:
#Ques2
run_query("""SELECT o.customer_id, o.order_id, o.sales FROM orders AS o
WHERE o.sales = (SELECT MAX(o2.sales) FROM orders AS o2 WHERE o2.customer_id = o.customer_id)
ORDER BY o.sales DESC;""")

,customer_id,order_id,sales
0,SM-20320,CA-2014-145317,22638.480
1,TC-20980,CA-2016-118689,17499.950
2,RB-19360,CA-2017-140151,13999.960
3,TA-21385,CA-2017-127180,11199.968
4,HL-15040,CA-2017-166709,10499.970
5,AB-10105,CA-2016-117121,9892.740
6,SC-20095,CA-2014-116904,9449.950
7,BS-11365,US-2016-107440,9099.930
8,SE-20110,CA-2016-158841,8749.950
9,CC-12370,CA-2016-143714,8399.976


In [212]:
#Ques3
run_query("""WITH customer_totals AS (
  SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT ct.customer_id, c.customer_name, ROUND(ct.total_sales,2) AS total_sales
FROM customer_totals AS ct
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers)AS c
  ON ct.customer_id = c.customer_id
ORDER BY total_sales DESC;""")

,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


In [213]:
#Ques4
run_query("""WITH customer_totals AS (
  SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT ct.customer_id, c.customer_name, ROUND(ct.total_sales,2) AS total_sales
FROM customer_totals AS ct
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c 
  ON ct.customer_id=c.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY total_sales DESC;""")

,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


In [214]:
#Ques5
run_query("""WITH customer_totals AS (
  SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT ct.customer_id, c.customer_name, ROUND(ct.total_sales,2) AS total_sales,
  RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
FROM customer_totals AS ct
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c
  ON ct.customer_id=c.customer_id
ORDER BY sales_rank;""")

,customer_id,customer_name,total_sales,sales_rank
0,SM-20320,Sean Miller,25043.05,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.34,3
3,TA-21385,Tom Ashbrook,14595.62,4
4,AB-10105,Adrian Barton,14473.57,5
5,KL-16645,Ken Lonsdale,14175.23,6
6,SC-20095,Sanjit Chand,14142.33,7
7,HL-15040,Hunter Lopez,12873.30,8
8,SE-20110,Sanjit Engle,12209.44,9
9,CC-12370,Christopher Conant,12129.07,10


In [215]:
#Ques6 
run_query("""SELECT customer_id, order_id, ROUND(sales,2) AS sales,
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) AS rn
FROM orders ORDER BY customer_id, rn;""")

,customer_id,order_id,sales,rn
0,AA-10315,CA-2016-103982,3930.07,1
1,AA-10315,CA-2014-128055,673.57,2
2,AA-10315,CA-2016-103982,431.98,3
3,AA-10315,CA-2017-147039,362.94,4
4,AA-10315,CA-2014-128055,52.98,5
5,AA-10315,CA-2016-103982,41.72,6
6,AA-10315,CA-2015-121391,26.96,7
7,AA-10315,CA-2014-138100,14.94,8
8,AA-10315,CA-2014-138100,14.56,9
9,AA-10315,CA-2017-147039,11.54,10


In [216]:
#Ques7
run_query("""WITH customer_totals AS (
  SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id),
ranked AS (
  SELECT customer_id, total_sales, RANK() OVER (ORDER BY total_sales DESC) AS rnk
  FROM customer_totals)
SELECT r.customer_id, c.customer_name, ROUND(r.total_sales,2) AS total_sales, r.rnk
FROM ranked AS r
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c 
  ON r.customer_id=c.customer_id
WHERE r.rnk <= 3 ORDER BY r.rnk;""")

,customer_id,customer_name,total_sales,rnk
0,SM-20320,Sean Miller,25043.05,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.34,3


# Step 4: Final Combined Query(Using JOIN + CTE + Window Function Together)
Write one final query that shows:
- Customer Name
- Total Sales 
- Rank

In [217]:
run_query("""WITH customer_totals AS (
  SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT c.customer_name AS "Customer Name", ROUND(ct.total_sales,2) AS "Total Sales",
  RANK() OVER (ORDER BY ct.total_sales DESC) AS "Rank"
FROM customer_totals ct
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) c ON ct.customer_id=c.customer_id
ORDER BY "Rank";""")

,Customer Name,Total Sales,Rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.23,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.30,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


# Mini Project: Customer Sales Insights
Answer the following using SQL: 
1. Who are the top 5 customers?  
2. Who are the bottom 5 customers?  
3. Which customers made only one order?  
4. Which customers have above-average sales?  
5. What is the highest order value per customer? 

In [218]:
#Ques1
run_query("""WITH t AS (SELECT customer_id, SUM(sales) total_sales FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(t.total_sales,2) total_sales FROM t
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c 
  ON t.customer_id=c.customer_id
ORDER BY t.total_sales DESC LIMIT 5;""")

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57


In [219]:
#Ques2
run_query("""WITH t AS (SELECT customer_id, SUM(sales) total_sales FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(t.total_sales,2) total_sales FROM t
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c 
  ON t.customer_id=c.customer_id
ORDER BY t.total_sales ASC LIMIT 5;""")

,customer_name,total_sales
0,Thais Sissman,4.83
1,Lela Donovan,5.30
2,Carl Jackson,16.52
3,Mitch Gastineau,16.74
4,Roy Skaria,22.33


In [220]:
#Ques3
run_query("""WITH oc AS (SELECT customer_id, COUNT(DISTINCT order_id) norders FROM orders GROUP BY customer_id)
SELECT c.customer_name, oc.norders FROM oc
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c
  ON oc.customer_id=c.customer_id
WHERE oc.norders = 1 ORDER BY c.customer_name;""")

,customer_name,norders
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


In [221]:
#Ques4
run_query("""WITH t AS (SELECT customer_id, SUM(sales) total_sales FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(t.total_sales,2) total_sales FROM t
JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c
  ON t.customer_id=c.customer_id
WHERE t.total_sales > (SELECT AVG(total_sales) FROM t) ORDER BY t.total_sales DESC;""")

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


In [222]:
#Ques5
run_query("""SELECT c.customer_name, ROUND(MAX(o.sales),2) highest_order_value
FROM orders AS o JOIN (SELECT DISTINCT customer_id, customer_name FROM customers) AS c
ON o.customer_id=c.customer_id GROUP BY o.customer_id, c.customer_name
ORDER BY highest_order_value DESC;""")

,customer_name,highest_order_value
0,Sean Miller,22638.48
1,Tamara Chand,17499.95
2,Raymond Buch,13999.96
3,Tom Ashbrook,11199.97
4,Hunter Lopez,10499.97
5,Adrian Barton,9892.74
6,Sanjit Chand,9449.95
7,Bill Shonely,9099.93
8,Sanjit Engle,8749.95
9,Christopher Conant,8399.98


## Key Insights
- **Sean Miller** is the highest-value customer (~$25,043 total sales), followed by Tamara Chand and Raymond Buch.
- **2,360** of 9,994 order line items exceed the average line-item sales value - a small share of large orders drives revenue (long-tail distribution).
- Only **294 of 793** customers post above-average total sales, confirming revenue concentration among a minority of customers.
- **12 customers** placed just a single order - candidates for re-engagement / retention campaigns.
- The lowest-value customer (Thais Sissman) contributed under **$5**, highlighting a long tail of low-spend customers.